In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": True, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import pandas as pd
import polars as pl
import util
from pathlib import Path

pd.set_option('display.float_format', '{:,.0f}'.format)

In [3]:
transit_jobs_access = pd.read_csv(util.output_path / 'access/transit_jobs_access.csv', 
                                  usecols=['geography', 'value', 'geography_group'])
walk_bike_jobs_access = pd.read_csv(util.output_path / 'access/walk_bike_jobs_access.csv', 
                                  usecols=['geography_value', 'jobs_1_mile_walk', 'jobs_3_mile_bike', 'geography_group']).\
                                  rename(columns={'geography_value': 'geography'})

parcel_emp = util.get_parcels_urbansim_data(inc_geog=True)

# jobs access in equity geographies
equity_geogs = util.summary_config['equity_geogs']

## Jobs Accessible within 45 Minutes of Transit

In [4]:
def process_access_data(jobs_access):
    df_access = jobs_access.copy()
    # rename region
    df_access.loc[jobs_access['geography_group'] == 'region', 'geography'] = 'Region'
    # rename rgc
    df_access.loc[jobs_access['geography_group'] == 'rgc_binary', 'geography'] = ['Not in RGC', 'In RGC']
    # rename regional geography
    df_access.loc[jobs_access['geography_group'] == 'rg_proposed', 'geography_group'] = 'RegionalGeogName'

    equity_geogs = list(util.equity_geog_dict.keys())
    df_access_equity = df_access.loc[
        df_access['geography_group'].isin(equity_geogs)].copy()
    
    df_access_equity['geography'] = df_access_equity['geography'].map({"0.0": 'Below Regional Average', 
                                                                       "1.0": 'Above Regional Average', 
                                                                       "2.0": 'Higher Share of Equity Population'}
                                                                       )
    df_access_equity['geography_group'] = df_access_equity['geography_group'].map(util.equity_geog_dict)

    # df_access_equity_geogs['geography'] = df_access_equity_geogs['geography_group']
    # df['geography'] = "NOT in " + df['geography_group']

    # df_access_equity_geogs = pd.concat([df_access_equity_geogs, df], ignore_index=True)
    # df_access_equity_geogs['geography_group'] = 'Equity Geography'

    return df_access, df_access_equity


df_access_t, df_access_equity_t = process_access_data(transit_jobs_access)
df_access_bp, df_access_equity_bp = process_access_data(walk_bike_jobs_access)
tot_jobs = parcel_emp['emptot_p'].sum()

In [5]:
def job_access_geog(access_table, geog, format=True):
    df = access_table.loc[access_table['geography_group'].isin([geog, 'region'])].\
        rename(columns={'value': 'Jobs within 45-minute Transit Commute'}).\
        drop(columns=['geography_group']).\
        set_index('geography')
    
    df_jobs = parcel_emp.groupby(geog)['emptot_p'].sum()
    df_jobs['Region'] = tot_jobs
    df = pd.concat([df, df_jobs.rename('Total Jobs within Geography')], axis=1)
    df = df.loc[df.index != 'Outside Region'].copy()

    df['% All Jobs'] = df['Jobs within 45-minute Transit Commute'] / tot_jobs
    df['% of Jobs in Geography'] = df['Jobs within 45-minute Transit Commute'] / df['Total Jobs within Geography']

    if format:
        return df.style.format('{:,.1%}', subset=['% All Jobs','% of Jobs in Geography']).\
            format('{:,.0f}', subset=['Jobs within 45-minute Transit Commute','Total Jobs within Geography'])
    else:
        return df


In [6]:
df = job_access_geog(df_access_t,'CountyName')
# df = df[df.index != 'Outside Region']
df

,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
King,"230,064","1,442,506",10.7%,15.9%
Kitsap,"7,072","98,821",0.3%,7.2%
Pierce,"18,896","323,128",0.9%,5.8%
Snohomish,"28,429","290,359",1.3%,9.8%
Region,"136,367","2,154,814",6.3%,6.3%


In [7]:
df_rgc = job_access_geog(df_access_t,'rgc_binary')
df_rgc

,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
Region,"136,367","2,154,814",6.3%,6.3%
Not in RGC,"100,926","1,399,649",4.7%,7.2%
In RGC,"470,084","755,165",21.8%,62.2%


In [8]:
df = job_access_geog(df_access_t,'GrowthCenterName')

df

,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
Region,"136,367","2,154,814",6.3%,6.3%
Auburn,"68,331","3,845",3.2%,"1,777.1%"
Bellevue,"481,018","52,050",22.3%,924.1%
Bothell Canyon Park,"46,377","13,717",2.2%,338.1%
Bremerton,"39,425","22,439",1.8%,175.7%
Burien,"124,528","3,758",5.8%,"3,313.7%"
Everett,"64,998","14,300",3.0%,454.5%
Federal Way,"104,332","2,358",4.8%,"4,424.6%"
Greater Downtown Kirkland,"236,484","11,284",11.0%,"2,095.7%"
Issaquah,0,"9,877",0.0%,0.0%


In [9]:
job_access_geog(df_access_t,'RegionalGeogName')


,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
Region,"136,367","2,154,814",6.3%,6.3%
Cities and Towns,"10,558","103,706",0.5%,10.2%
Core Cities,"58,724",nan,2.7%,nan%
High Capacity Transit Communities,"37,870",nan,1.8%,nan%
Metropolitan Cities,"335,662",nan,15.6%,nan%
Rural Areas,"1,851",nan,0.1%,nan%
Urban Unincorporated Areas,"10,475",nan,0.5%,nan%


In [10]:
df = pd.DataFrame()
for col, label in util.equity_geog_dict.items():
    _df = job_access_geog(df_access_equity_t, label, format=False)
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df.rename(columns={'index': 'EFA Type'}, inplace=True)
# df
df.loc[df['EFA Type']!="Region"][['Group', 'EFA Type', 'Jobs within 45-minute Transit Commute',
    'Total Jobs within Geography', '% All Jobs', '% of Jobs in Geography']].style.\
        format('{:,.1%}', subset=['% All Jobs','% of Jobs in Geography']).\
        format('{:,.0f}', subset=['Jobs within 45-minute Transit Commute','Total Jobs within Geography'])

,Group,EFA Type,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
0,People of Color,Below Regional Average,"119,168","823,085",5.5%,14.5%
1,People of Color,Above Regional Average,"154,336","762,693",7.2%,20.2%
2,People of Color,Higher Share of Equity Population,"160,976","569,036",7.5%,28.3%
4,Income,Below Regional Average,"137,417","1,072,742",6.4%,12.8%
5,Income,Above Regional Average,"128,134","616,977",5.9%,20.8%
6,Income,Higher Share of Equity Population,"146,720","465,095",6.8%,31.5%
8,LEP,Below Regional Average,"143,362","1,156,089",6.7%,12.4%
9,LEP,Above Regional Average,"126,452","510,608",5.9%,24.8%
10,LEP,Higher Share of Equity Population,"122,057","488,117",5.7%,25.0%
12,Disability,Below Regional Average,"157,411","1,070,762",7.3%,14.7%


## Average Jobs Accessible within 1 Mile Walk and 3 Mile Bike
Note that this is not using the bike network, but is instead using the all-streets network.

Average accessible jobs are weighted averages based on parcel household population.

In [11]:
def bp_job_access_geog(access_table,geog):
    df = access_table.loc[access_table['geography_group'].isin(['region', geog])].\
        rename(columns={'jobs_1_mile_walk': 'Jobs within 1-mile Walk',
                        'jobs_3_mile_bike': 'Jobs within 3-mile Bike'}).\
        drop(columns=['geography_group']).\
        set_index('geography')

    df['% Total Jobs (1-mile walk)'] = df['Jobs within 1-mile Walk'].apply(lambda x: f'{x / tot_jobs * 100:,.1f}' + '%')
    df['% Total Jobs (3-mile bike)'] = df['Jobs within 3-mile Bike'].apply(lambda x: f'{x / tot_jobs * 100:,.1f}' + '%')

    return df

In [12]:
df = bp_job_access_geog(df_access_bp,'CountyName')
df = df[df.index != 'Outside Region']
df

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
geography,,,,
King,"18,516","84,913",0.9%,3.9%
Kitsap,"1,236","8,014",0.1%,0.4%
Pierce,"2,508","18,163",0.1%,0.8%
Snohomish,"2,006","17,703",0.1%,0.8%
Region,"11,164","54,254",0.5%,2.5%


In [13]:
df_rgc = bp_job_access_geog(df_access_bp,'rgc_binary')
df = bp_job_access_geog(df_access_bp,'GrowthCenterName')

pd.concat([df_rgc, df.loc[~df.index.isin(['Region','Not in RGC'])]], axis=0)

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
geography,,,,
Region,"11,164","54,254",0.5%,2.5%
Not in RGC,"2,912","34,869",0.1%,1.6%
In RGC,"88,864","236,789",4.1%,11.0%
Auburn,"10,541","40,324",0.5%,1.9%
Bellevue,"59,188","110,699",2.7%,5.1%
Bothell Canyon Park,"8,539","21,748",0.4%,1.0%
Bremerton,"11,536","34,387",0.5%,1.6%
Burien,"4,829","13,400",0.2%,0.6%
Everett,"15,908","39,722",0.7%,1.8%


In [14]:
bp_job_access_geog(df_access_bp,'RegionalGeogName')

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
geography,,,,
Region,"11,164","54,254",0.5%,2.5%
Cities and Towns,984,"8,948",0.0%,0.4%
Core Cities,"3,415","28,587",0.2%,1.3%
High Capacity Transit Communities,"1,428","15,237",0.1%,0.7%
Metropolitan Cities,"29,432","128,074",1.4%,5.9%
Rural Areas,142,"2,387",0.0%,0.1%
Urban Unincorporated Areas,443,"6,816",0.0%,0.3%


In [15]:
df = pd.DataFrame()
for col, label in util.equity_geog_dict.items():
    _df = bp_job_access_geog(df_access_equity_bp, label)
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df.rename(columns={'geography': 'EFA Type'}, inplace=True)
df[['Group', 'EFA Type', 'Jobs within 1-mile Walk',	'Jobs within 3-mile Bike',	'% Total Jobs (1-mile walk)','% Total Jobs (3-mile bike)']]

,Group,EFA Type,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
0,People of Color,Below Regional Average,"5,374","41,935",0.2%,1.9%
1,People of Color,Above Regional Average,"18,509","71,731",0.9%,3.3%
2,People of Color,Higher Share of Equity Population,"17,229","63,994",0.8%,3.0%
3,Income,Below Regional Average,"9,394","52,420",0.4%,2.4%
4,Income,Above Regional Average,"13,899","56,258",0.6%,2.6%
5,Income,Higher Share of Equity Population,"13,651","58,320",0.6%,2.7%
6,LEP,Below Regional Average,"11,368","57,818",0.5%,2.7%
7,LEP,Above Regional Average,"13,284","51,842",0.6%,2.4%
8,LEP,Higher Share of Equity Population,"7,745","43,674",0.4%,2.0%
9,Disability,Below Regional Average,"9,651","56,870",0.4%,2.6%


## Intersection Density

In [16]:
buffered_parcels = pl.read_csv(util.output_path / 'landuse/buffered_parcels.txt', 
                               separator=' ',
                               columns=['parcelid','nodes3_2','nodes4_2','hh_p']).to_pandas()


list_cols = ['ParcelID','CountyName','GrowthCenterName','rg_proposed'] + equity_geogs
parcel_geog = util.get_parcel_geog()[list_cols]

In [17]:
df_intersection = buffered_parcels.merge(parcel_geog, left_on='parcelid', right_on='ParcelID')

# Total intersections within 1/2 mile buffer
df_intersection['intersections_wt'] = (df_intersection['nodes3_2'] + df_intersection['nodes4_2']) * df_intersection['hh_p']

In [18]:
def intersection_density(geog, map=False):
    df = df_intersection.groupby(geog)[['intersections_wt', 'hh_p']].sum().reset_index()
    df['Intersections'] = df['intersections_wt']/df['hh_p']

    if map:
        df[geog] = df[geog].astype('int').map({0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}
                                )
    
    return df[[geog] + ['Intersections']]

In [19]:
df = intersection_density('CountyName')
df = df[df['CountyName']!='Outside Region']
df

,CountyName,Intersections
0,King,155
1,Kitsap,51
3,Pierce,80
4,Snohomish,73


In [20]:
intersection_density('GrowthCenterName')

,GrowthCenterName,Intersections
0,Auburn,192
1,Bellevue,273
2,Bothell Canyon Park,69
3,Bremerton,168
4,Burien,176
5,Everett,161
6,Federal Way,134
7,Greater Downtown Kirkland,167
8,Issaquah,NaN
9,Kent,214


In [21]:
intersection_density('rg_proposed')

,rg_proposed,Intersections
0,Cities and Towns,65
1,Core Cities,103
2,High Capacity Transit Communities,84
3,Metropolitan Cities,202
4,Rural Areas,20
5,Urban Unincorporated Areas,56


In [22]:
df = pd.DataFrame()
for col, label in util.equity_geog_dict.items():
    _df = intersection_density(col).rename(columns={col: "Group"})
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df['EFA Type'] = df['index'].map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })
df[['Group', 'EFA Type', 'Intersections']]

,Group,EFA Type,Intersections
0,People of Color,Below Regional Average,103
1,People of Color,Above Regional Average,136
2,People of Color,Higher Share of Equity Population,136
3,Income,Below Regional Average,112
4,Income,Above Regional Average,122
5,Income,Higher Share of Equity Population,137
6,LEP,Below Regional Average,117
7,LEP,Above Regional Average,120
8,LEP,Higher Share of Equity Population,121
9,Disability,Below Regional Average,121
